In [1]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv()
HEADERS = {"User-Agent": os.getenv("SEC_USER_AGENT")}

# Ridgeline's universe: SIC codes chosen in Phase 4, Step 8
UNIVERSE_SIC_CODES = [7320, 7340, 7381, 7389, 7600, 8711, 8742]

url = "https://www.sec.gov/files/company_tickers_exchange.json"
response = requests.get(url, headers=HEADERS, timeout=30)
print("Status code:", response.status_code)

raw = response.json()
listed = pd.DataFrame(raw["data"], columns=raw["fields"])
print("All tickers:", len(listed))

# Keep only the two major U.S. exchanges
listed = listed[listed["exchange"].isin(["NYSE", "Nasdaq"])]

# Some companies have several tickers (share classes); keep one row per company
listed = listed.drop_duplicates(subset="cik")
print("Companies on NYSE or Nasdaq:", len(listed))

listed.head()

Status code: 200
All tickers: 10429
Companies on NYSE or Nasdaq: 6041


,cik,name,ticker,exchange
0,1045810,NVIDIA CORP,NVDA,Nasdaq
1,320193,Apple Inc.,AAPL,Nasdaq
2,1652044,Alphabet Inc.,GOOGL,Nasdaq
3,789019,MICROSOFT CORP,MSFT,Nasdaq
4,1018724,AMAZON COM INC,AMZN,Nasdaq


In [2]:
cik = 1436126   # Mistras Group
url = f"https://data.sec.gov/submissions/CIK{cik:010d}.json"
profile = requests.get(url, headers=HEADERS, timeout=30).json()

print("Name:", profile["name"])
print("SIC:", profile["sic"], "-", profile["sicDescription"])
print("Fiscal year end:", profile["fiscalYearEnd"])
print("State:", profile["addresses"]["business"]["stateOrCountry"])

Name: Mistras Group, Inc.
SIC: 8711 - Services-Engineering Services
Fiscal year end: 1231
State: NJ


In [3]:
import time

os.makedirs("../data/raw", exist_ok=True)
PROFILE_FILE = "../data/raw/company_profiles.csv"

# Resume: skip companies already saved from an earlier run
if os.path.exists(PROFILE_FILE):
    done_ciks = set(pd.read_csv(PROFILE_FILE)["cik"])
else:
    done_ciks = set()

to_fetch = listed[~listed["cik"].isin(done_ciks)]
print("Already saved:", len(done_ciks), "| To download:", len(to_fetch))

rows = []
for i, (cik, ticker) in enumerate(zip(to_fetch["cik"], to_fetch["ticker"]), start=1):
    url = f"https://data.sec.gov/submissions/CIK{cik:010d}.json"
    try:
        p = requests.get(url, headers=HEADERS, timeout=30).json()
        rows.append({
            "cik": cik,
            "ticker": ticker,
            "company_name": p.get("name"),
            "sic_code": p.get("sic"),
            "sic_description": p.get("sicDescription"),
            "fiscal_year_end": p.get("fiscalYearEnd"),
            "hq_state": p["addresses"]["business"]["stateOrCountry"],
        })
    except Exception as error:
        print("Failed:", cik, ticker, error)

    time.sleep(0.12)   # stay under the SEC's rate limit

    # Save every 250 companies, and at the very end
    if i % 250 == 0 or i == len(to_fetch):
        if rows:
            pd.DataFrame(rows).to_csv(PROFILE_FILE, mode="a",
                                      header=not os.path.exists(PROFILE_FILE), index=False)
        rows = []
        print(f"Saved {i} of {len(to_fetch)}")

Already saved: 0 | To download: 6041
Saved 250 of 6041
Saved 500 of 6041
Saved 750 of 6041
Saved 1000 of 6041
Saved 1250 of 6041
Saved 1500 of 6041
Saved 1750 of 6041
Saved 2000 of 6041
Saved 2250 of 6041
Saved 2500 of 6041
Saved 2750 of 6041
Saved 3000 of 6041
Saved 3250 of 6041
Saved 3500 of 6041
Saved 3750 of 6041
Saved 4000 of 6041
Saved 4250 of 6041
Saved 4500 of 6041
Saved 4750 of 6041
Saved 5000 of 6041
Saved 5250 of 6041
Saved 5500 of 6041
Saved 5750 of 6041
Saved 6000 of 6041
Saved 6041 of 6041


In [4]:
profiles = pd.read_csv(PROFILE_FILE)
print("Profiles saved:", len(profiles))

universe = profiles[profiles["sic_code"].isin(UNIVERSE_SIC_CODES)]
print("Companies in Ridgeline's SIC codes:", len(universe))

universe["sic_code"].value_counts()

Profiles saved: 6041
Companies in Ridgeline's SIC codes: 167


sic_code
7389.0    105
8742.0     34
8711.0      9
7340.0      7
7381.0      6
7320.0      4
7600.0      2
Name: count, dtype: int64

In [5]:
   universe[universe["ticker"] == "MG"]

,cik,ticker,company_name,sic_code,sic_description,fiscal_year_end,hq_state
2813,1436126,MG,"Mistras Group, Inc.",8711.0,Services-Engineering Services,1231.0,NJ


In [6]:
os.makedirs("../data/processed", exist_ok=True)
universe.to_csv("../data/processed/universe_sic_filtered.csv", index=False)
print("Saved", len(universe), "companies")

Saved 167 companies
